## Notebook for Plotting Turtle Sensor Data

In [1]:
#!/usr/bin/env python3
import sys
import os
import glob
import re
import cv2
import numpy as np
import matplotlib
matplotlib.use('TkAgg')  # Try this backend for displaying plots
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.widgets import Slider, Button
from tqdm import tqdm  # For progress bars (install with pip install tqdm if needed)
import transforms3d.euler as euler
print(f"OpenCV version: {cv2.__version__}")


OpenCV version: 4.12.0


### Turtle Data Keys

*Shape format: (timesteps, features) where n = number of samples*

- `q` (n, 10) — Joint positions (radians)
- `dq` (n, 10) — Joint velocities (rad/s)
- `t` (n,) — Timestamps (seconds)
- `nav_u` (n, 4) — Navigation control signals from planner

    - [velocity, roll, pitch for front flippers, yaw, pitch for rear flippers]
    - technically it should be (n,5) but we don't record the pitch for rear flippers
- `u` (n, 10) — Actuator motor control signal (radians)

    - has to be passed into grab_arm_current() to ensure we pass in safe currents to the motors
- `input` (n, 10) — Motor current commands (dynamixel units)

    - what is actually passed into the dynamixel motors
- `qd` (n, 10) — Desired joint positions (radians)
- `dqd` (n, 10) — Desired joint velocities (rad/s)
- `depth` (n,) — Depth measurements (meters)
- `depth_d` (n,) — Desired depth (meters)
- `quat` (n, 4) — Orientation quaternion (unitless)
- `alt` (n,) — Altitude (meters)
- `yaw_d` (n,) — Desired yaw angle (radians)
- `stereo_depth` (n,) — Stereo camera depth (meters)
- `stereo_point` (2, n) — Stereo point coordinates (pixels)

### Helper Functions

In [2]:
def natural_sort_key(s):
    """
    Sort strings containing numbers naturally (e.g., frame1.npy, frame2.npy, ..., frame10.npy)
    """
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def load_depth_files(directory_path):
    """Load and sort all depth .npy files from the given directory"""
    if not os.path.exists(directory_path):
        print(f"Error: Directory '{directory_path}' not found.")
        return []
        
    # Find all .npy files in the directory
    depth_files = glob.glob(os.path.join(directory_path, "*.npy"))
    
    if not depth_files:
        print(f"Error: No .npy files found in '{directory_path}'")
        return []
    
    # Sort files naturally (so frame10 comes after frame9, not after frame1)
    depth_files.sort(key=natural_sort_key)
    # print(f"Found {len(depth_files)} depth data files.")
    
    return depth_files


### Open file path of turtle data

In [4]:
file_path = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/data/09_04_2025_data_NEA/04_16_2025_03_03_18_tetherless_A/np_data_04_16_2025_03_03_18.npz'
gen_fpath = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/data/09_04_2025_data_NEA/04_16_2025_03_03_18_tetherless_A'

file_path = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/tetherless_trial_C_data_sept_30_11_pm.npz'
gen_fpath = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C'

# file_path = '/home/ranger/crush_tetherless_tracking_videos_december_18/data/12_18_2025_10_46_30/np_data_12_18_2025_10_46_30.npz'
# gen_fpath = '/home/ranger/crush_tetherless_tracking_videos_december_18/data/12_18_2025_10_46_30'
# file_path = '/home/ranger/11_20_2025_17_52_56_tetherless_OA_trial_1_depth_d_0.1/np_data_11_20_2025_17_52_56.npz'
# gen_fpath = '/home/ranger/11_20_2025_17_52_56_tetherless_OA_trial_1_depth_d_0.1'

# gen_fpath = '/home/ranger/04_14_2025_05_06_13_success'
# file_path = '/home/ranger/Downloads/08_19_2025_10_26_41/08_19_2025_10_26_41_np_config.npz'
# gen_fpath = '/home/ranger/Downloads/08_19_2025_10_26_41'
# file_path = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_030311_tetherless_A/tetherless_a_other_logger.npz'
# gen_fpath = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_030311_tetherless_A'


# file_path = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_030830_tetherless_B/tetherless_b_other_logger.npz'
# gen_fpath = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_030830_tetherless_B'

# file_path = '/home/ranger/04_13_2025_01_55_41_trial_1/np_data_04_13_2025_01_55_41.npz'
# gen_fpath = '/home/ranger/04_13_2025_01_55_41_trial_1'

# file_path = '/home/ranger/04_13_2025_02_26_49_trial_2/np_data_04_13_2025_02_26_49.npz'
# gen_fpath = '/home/ranger/04_13_2025_02_26_49_trial_2'

# file_path = '/home/ranger/04_13_2025_05_47_20_check_navu4/np_data_04_13_2025_05_47_20.npz'
# gen_fpath = '/home/ranger/04_13_2025_05_47_20_check_navu4'
# Load the NPZ file
npz_file = np.load(file_path, allow_pickle=True)
keys = list(npz_file.keys())
print(f"Keys found: {keys}")
for key in keys:
    print(f"{key} has shape: {npz_file[key].shape}")
# Create a directory to save plots
plot_dir = os.path.join(os.path.dirname(file_path), "plots")
os.makedirs(plot_dir, exist_ok=True)
print(f"Saving plots to: {plot_dir}")

Keys found: ['q', 'dq', 't_sensor_node', 't_logger_node', 'u', 'nav_u', 'qd', 'dqd', 'depth', 'depth_d', 'quat', 'alt', 'yaw_d', 'stereo_depth', 'stereo_debug_depth', 'stereo_point', 'acc', 'gyr', 'voltage', 'fps', 'headings', 'turn_commands']
q has shape: (21713, 10)
dq has shape: (21713, 10)
t_sensor_node has shape: (21713,)
t_logger_node has shape: (21713,)
u has shape: (35986, 10)
nav_u has shape: (8422, 4)
qd has shape: (21713, 10)
dqd has shape: (21713, 10)
depth has shape: (21713,)
depth_d has shape: (21713,)
quat has shape: (21713, 4)
alt has shape: (21713,)
yaw_d has shape: (8423,)
stereo_depth has shape: (1163,)
stereo_debug_depth has shape: (8422,)
stereo_point has shape: (2, 21713)
acc has shape: (21713, 3)
gyr has shape: (21713, 3)
voltage has shape: (21713,)
fps has shape: (1163,)
headings has shape: (8422,)
turn_commands has shape: (8422,)
Saving plots to: /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031

In [6]:
num_depths = len(npz_file['depth'])
# for i in range(num_depths):
#     print(npz_file['depth'][i])

### Optionally open up timestamps of the images if provided


In [10]:
print("======== time stamps from timestamps_C.txt ============")
timestamps_file = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/timestamps_C.txt'
# for tetherless C, the timestamps file is timestamps_C.txt which include the damaged frames (therefore total count is 1450 vs actual 1336 frames saved)
data = np.genfromtxt(timestamps_file, delimiter=',', skip_header=1)
timestamps_unix = data[:, 1]
t_elapsed_frames = timestamps_unix - timestamps_unix[0]
print(t_elapsed_frames[0:30])
print(f"end time: {t_elapsed_frames[-1]}")
print(f"number of elapsed frames: {t_elapsed_frames.shape}")

======== time stamps from timestamps_C.txt ============
[0.         0.39900017 0.74600005 0.97000003 1.22500014 1.49800014
 1.74300003 2.09000015 2.35400009 2.90200019 3.35800004 3.72800016
 3.9920001  4.23700023 4.53500009 4.76900005 5.00100017 5.22900009
 5.47500014 5.77800012 6.01300001 6.31500006 6.671      6.98500013
 7.35000014 7.7750001  8.17900014 8.50600004 8.82000017 9.13900018]
end time: 453.507000207901
number of elapsed frames: (1450,)


In [12]:
# from matching quaternion and headings
print("========== timestamps form matching quaternion and headings ==========")
timestamps_file = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/matched_timestamps.txt'
data = np.genfromtxt(timestamps_file, delimiter=',', skip_header=1)
timestamps_unix = data
t_quat_heading = timestamps_unix - timestamps_unix[0]
print(t_quat_heading[0:30])
print(f"end time: {t_quat_heading[-1]}")
print(f"number of t points: {t_quat_heading.shape}")

========== timestamps form matching quaternion and headings ==========
[0.       1.026418 1.062439 1.109886 1.160863 1.222217 1.247823 1.300995
 1.358367 1.416128 1.467357 1.495533 1.568904 1.608419 1.63521  1.712717
 1.754161 1.811457 1.853546 1.904652 1.965189 2.009032 2.050366 2.092936
 2.174034 2.222114 2.243631 2.308233 2.346382 2.391921]
end time: 422.013517
number of t points: (8422,)


In [23]:
print(npz_file['turn_commands'][0:10])
# t logger has no timestamps
# print(f"num logger time points: {npz_file['t_logger_node'].shape}")
print(f"num sendor time points: {npz_file['t_sensor_node'].shape}")

['no turn' 'no turn' 'no turn' 'no turn' 'no turn' 'no turn' 'no turn'
 'no turn' 'no turn' 'no turn']
num sendor time points: (21713,)


In [24]:
import pandas as pd
output_file_path = os.path.join(gen_fpath, "stereo_depth_data.csv")
stereo_depth_data = npz_file['stereo_depth']
stereo_points = npz_file['stereo_point']
print(f"stereo_depth_data shape: {stereo_depth_data.shape}")
print(f"stereo_points shape: {stereo_points[:,1].shape}")
# print(stereo_depth_data[350:1500])
# if stereo_depth_data.ndim == 1:
#     # 1D array - simple conversion
#     df = pd.DataFrame({
#         'Index': range(len(stereo_depth_data)),
#         'Depth_Value': stereo_depth_data,
#         'Point_X': stereo_points[0, :],
#         'Point_Y': stereo_points[1, :],
#     })
#     df.to_csv(output_file_path, index=False)
#     print(f"\nData saved to CSV file: {output_file_path}")


stereo_depth_data shape: (1163,)
stereo_points shape: (2,)


## Motor Current Input Plotting
### Motor input (u)

In [63]:
print(npz_file['u'].shape)
u_data = npz_file['u']
t_data = npz_file['t_logger_node']

# Check if t_logger_node needs elapsed time conversion
if t_data[0] > 1e9:  # Unix timestamp format
    t_elapsed = t_data - t_data[0]
else:
    t_elapsed = t_data

# Handle length mismatch - use the shorter length
min_len = min(len(u_data), len(t_elapsed))
u_data = u_data[:min_len]
t_elapsed = t_elapsed[:min_len]

# Find indices for time range 7.77 to 9.13 seconds
time_start = 5
time_end = 15
# start_idx = np.argmin(np.abs(t_elapsed - time_start))
# end_idx = np.argmin(np.abs(t_elapsed - time_end))

start_idx = 0
end_idx = 1200

print(f"Plotting from index {start_idx} to {end_idx}")
print(f"Time range: {t_elapsed[start_idx]:.2f}s to {t_elapsed[end_idx]:.2f}s")

x = t_elapsed[start_idx:end_idx]
x_label = 'Time (seconds)'


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Top plot: Individual motors
ax1.plot(x, u_data[start_idx:end_idx, 0], 'b-', label='u1 (Left motor)', linewidth=1.5, alpha=0.7)
ax1.plot(x, u_data[start_idx:end_idx, 3], 'r-', label='u4 (Right motor)', linewidth=1.5, alpha=0.7)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax1.set_ylabel('Motor Command')
ax1.set_title('Individual Motor Commands')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Bottom plot: Differential (u1 - u4)
u_diff = abs(u_data[start_idx:end_idx, 0]) - abs(u_data[start_idx:end_idx, 3])
ax2.plot(x, u_diff, 'g-', label='u1 - u4 (Turning moment)', linewidth=2)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.fill_between(x, 0, u_diff, where=(u_diff>0), alpha=0.3, color='green', label='Right turn')
ax2.fill_between(x, 0, u_diff, where=(u_diff<0), alpha=0.3, color='orange', label='Left turn')
ax2.set_xlabel('Time (seconds)')
ax2.set_ylabel('Differential Thrust')
ax2.set_title('Turning Moment (Positive = Right Turn)')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_path = os.path.join(plot_dir, f"turn_maneuver_commands.png")
plt.savefig(save_path, dpi=300)


# fig, ax = plt.subplots(figsize=(12, 7))

# # Plot both motors
# ax.plot(x, u_data[start_idx:end_idx, 0], 'b-', label='u1 (Right motor - High thrust)', linewidth=1.5)
# ax.plot(x, u_data[start_idx:end_idx, 3], 'r-', label='u4 (Left motor - Low thrust)', linewidth=1.5)

# ax.set_xlabel('Time (seconds)')
# ax.set_ylabel('Motor Command')
# ax.set_title(f'Right Turn Maneuver: Differential Thrust (Time: {time_start:.2f}-{time_end:.2f}s)')
# ax.legend(loc='best')
# ax.grid(True, alpha=0.3)
# ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)

# plt.tight_layout()
# save_path = os.path.join(plot_dir, f"turn_maneuver_u1_vs_u4.png")
# plt.savefig(save_path, dpi=300)



# for i in range(10):
#     fig, ax = plt.subplots(figsize=(12, 7))
    
#     ax.plot(x, u_data[start_idx:end_idx, i], 'b-', label=f'u{i+1}', linewidth=1.5)
#     # Add annotation for motor 1
#     if i == 0:  # motor 1
#         ax.text(0.5, 0.95, 'Higher amplitude → Right turn maneuver', 
#             transform=ax.transAxes, fontsize=10, 
#             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
#     ax.set_xlabel(x_label)
#     ax.set_ylabel('Motor Command')
#     ax.set_title(f'Motor Command u{i+1} (Time: {time_start:.2f}-{time_end:.2f}s)')
#     ax.legend(loc='best')
#     ax.grid(True, alpha=0.3)
    
#     plt.tight_layout()
    
#     save_path = os.path.join(plot_dir, f"u_motor_{i+1}_zoom.png")
#     plt.savefig(save_path, dpi=300)
#     print(f"Saved plot for motor {i+1}")
#     plt.close(fig)

(35986, 10)
Plotting from index 0 to 1200
Time range: 0.00s to 0.00s


In [ ]:
print(npz_file['u'].shape)
u_data = npz_file['u']
t_data = npz_file['t_logger_node']
# t_logger_node has shape: (34937,)
# u has shape: (35986, 10)

# Define range to plot
start_idx = 0
end_idx = t_data.shape[0]

# Prepare x-axis data
if 't' in npz_file:
    t_data = npz_file['t']
    if len(t_data) == len(u_data):
        x = t_data[start_idx:end_idx]
        x_label = 'Time (seconds)'
    else:
        x = np.arange(start_idx, end_idx)
        x_label = 'Index'
else:
    x = np.arange(start_idx, end_idx)
    x_label = 'Time (seconds)'

for i in range(10):
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Plot both q and dqd on same axis
    ax.plot(x, u_data[start_idx:end_idx, i], 'b-', label=f'u{i+1} (Desired Position)', linewidth=0.5)
    
    # Add labels and title
    ax.set_xlabel(x_label)
    ax.set_ylabel('radians')
    ax.set_title(f'Dimension {i+1}')
    
    # Add legend
    ax.legend(loc='best')
    
    # Add grid for readability
    ax.grid(True, alpha=0.3)
    
    # Calculate y-axis limits to ensure both datasets are visible
    y_min = min(np.min(u_data[start_idx:end_idx, i]), np.min(u_data[start_idx:end_idx, i]))
    y_max = max(np.max(u_data[start_idx:end_idx, i]), np.max(u_data[start_idx:end_idx, i]))
    
    # Add some padding to the y-axis limits
    y_range = y_max - y_min
    y_min -= 0.1 * y_range
    y_max += 0.1 * y_range
    
    ax.set_ylim(y_min, y_max)
    
    plt.tight_layout()
    
    # Save the plot
    save_path = os.path.join(plot_dir, f"u_motor_{i+1}.png")
    plt.savefig(save_path, dpi=300)
    print(f"Saved plot for dimension {i+1} to {save_path}")
    
    # Close the plot to free memory
    plt.close(fig)


(35986, 10)


## Motor Position Plotting
### Motor Positions and Desired Positions (q and dq)

In [9]:
# Get the data
q_data = npz_file['q']
qd_data = npz_file['qd']

print(f"q data shape: {q_data.shape}")
print(f"qd data shape: {qd_data.shape}")
    
# Define range to plot
start_idx = 0
end_idx = q_data.shape[0]

# Prepare x-axis data
if 't' in npz_file:
    t_data = npz_file['t']
    if len(t_data) == len(q_data):
        x = t_data[start_idx:end_idx]
        x_label = 'Time (seconds)'
    else:
        x = np.arange(start_idx, end_idx)
        x_label = 'Index'
else:
    x = np.arange(start_idx, end_idx)
    x_label = 'Index'

# Create individual plots for each dimension
for i in range(10):
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Plot both q and dqd on same axis
    ax.plot(x, q_data[start_idx:end_idx, i], 'b-', label=f'q{i+1} (Position)', linewidth=0.8)
    ax.plot(x, qd_data[start_idx:end_idx, i], 'r-', label=f'qd{i+1} (Desired Position)', linewidth=0.8)
    
    # Add labels and title
    ax.set_xlabel(x_label)
    ax.set_ylabel('radians')
    # ax.set_title(f'Dimension {i+1}: Position and Desired Position')
    
    # Add legend
    ax.legend(loc='best')
    
    # Add grid for readability
    ax.grid(True, alpha=0.3)
    
    # Calculate y-axis limits to ensure both datasets are visible
    y_min = min(np.min(q_data[start_idx:end_idx, i]), np.min(qd_data[start_idx:end_idx, i]))
    y_max = max(np.max(q_data[start_idx:end_idx, i]), np.max(qd_data[start_idx:end_idx, i]))
    
    # Add some padding to the y-axis limits
    y_range = y_max - y_min
    y_min -= 0.1 * y_range
    y_max += 0.1 * y_range
    
    ax.set_ylim(y_min, y_max)
    
    plt.tight_layout()
    
    # Save the plot
    save_path = os.path.join(plot_dir, f"position_motor_{i+1}.png")
    plt.savefig(save_path, dpi=300)
    print(f"Saved plot for dimension {i+1} to {save_path}")
    
    # Close the plot to free memory
    plt.close(fig)

# Create a combined figure with all dimensions (5x2 grid)
fig, axes = plt.subplots(5, 2, figsize=(18, 24))
axes = axes.flatten()

for i in range(10):
    ax = axes[i]
    
    # Plot both q and dqd on same axis
    # ax.plot(x, q_data[start_idx:end_idx, i], 'b-', label=f'q{i+1}', linewidth=1.2)
    ax.plot(x, qd_data[start_idx:end_idx, i], 'r-', label=f'qd{i+1}', linewidth=0.8)
    
    # Add labels and title
    ax.set_xlabel(x_label)
    ax.set_ylabel('radians')
    ax.set_title(f'Motor {i+1}')
    
    # Add legend
    ax.legend(loc='best', fontsize='small')
    
    # Add grid for readability
    ax.grid(True, alpha=0.3)
    
    # Calculate y-axis limits
    y_min = min(np.min(q_data[start_idx:end_idx, i]), np.min(qd_data[start_idx:end_idx, i]))
    y_max = max(np.max(q_data[start_idx:end_idx, i]), np.max(qd_data[start_idx:end_idx, i]))
    
    # Add padding
    y_range = y_max - y_min
    y_min -= 0.1 * y_range
    y_max += 0.1 * y_range
    
    ax.set_ylim(y_min, y_max)

plt.tight_layout()
# Save the combined plot
combined_path = os.path.join(plot_dir, "all_positions.png")
plt.savefig(combined_path, dpi=300)
print(f"Saved plot to {combined_path}")
plt.close()


q data shape: (21713, 10)
qd data shape: (21713, 10)


/tmp/ipykernel_482026/3854008609.py:53: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set_ylim(y_min, y_max)


Saved plot for dimension 1 to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/position_motor_1.png
Saved plot for dimension 2 to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/position_motor_2.png
Saved plot for dimension 3 to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/position_motor_3.png
Saved plot for dimension 4 to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/position_motor_4.png
Saved plot for dimension 5 to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/position_motor_5.png
Saved plot for dimension 6 to /home/ranger/drl-tur

/tmp/ipykernel_482026/3854008609.py:96: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set_ylim(y_min, y_max)


Saved plot to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/all_positions.png


## Velocity Plotting
### Velocities and Desired Velocities (dq and dqd)

In [9]:
# Get the data
dq_data = npz_file['dq']
dqd_data = npz_file['dqd']

print(f"dq data shape: {dq_data.shape}")
print(f"dqd data shape: {dqd_data.shape}")
    
# Define range to plot
start_idx = 0
end_idx = dq_data.shape[0]

# Prepare x-axis data
if 't' in npz_file:
    t_data = npz_file['t']
    if len(t_data) == len(q_data):
        x = t_data[start_idx:end_idx]
        x_label = 'Time (seconds)'
    else:
        x = np.arange(start_idx, end_idx)
        x_label = 'Index'
else:
    x = np.arange(start_idx, end_idx)
    x_label = 'Index'

# Create individual plots for each dimension
for i in range(10):
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Plot both q and dqd on same axis
    ax.plot(x, dq_data[start_idx:end_idx, i], 'b-', label=f'dq{i+1} (Velocity)', linewidth=1.5)
    ax.plot(x, dqd_data[start_idx:end_idx, i], 'r-', label=f'dqd{i+1} (Desired Velocity)', linewidth=1.5)
    
    # Add labels and title
    ax.set_xlabel(x_label)
    ax.set_ylabel(r'rad $\cdot$ sec$^{-1}$')
    ax.set_title(f'Motor {i+1}: Velocity and Desired Velocity')
    
    # Add legend
    ax.legend(loc='best')
    
    # Add grid for readability
    ax.grid(True, alpha=0.3)
    
    # Calculate y-axis limits to ensure both datasets are visible
    y_min = min(np.min(dq_data[start_idx:end_idx, i]), np.min(dqd_data[start_idx:end_idx, i]))
    y_max = max(np.max(dq_data[start_idx:end_idx, i]), np.max(dqd_data[start_idx:end_idx, i]))
    
    # Add some padding to the y-axis limits
    y_range = y_max - y_min
    y_min -= 0.1 * y_range
    y_max += 0.1 * y_range
    
    ax.set_ylim(y_min, y_max)
    
    plt.tight_layout()
    
    # Save the plot
    save_path = os.path.join(plot_dir, f"velocity_motor_{i+1}.png")
    plt.savefig(save_path, dpi=300)
    print(f"Saved plot for dimension {i+1} to {save_path}")
    
    # Close the plot to free memory
    plt.close(fig)

# Create a combined figure with all dimensions (5x2 grid)
fig, axes = plt.subplots(5, 2, figsize=(18, 24))
axes = axes.flatten()

for i in range(10):
    ax = axes[i]
    
    # Plot both q and dqd on same axis
    ax.plot(x, q_data[start_idx:end_idx, i], 'b-', label=f'dq{i+1}', linewidth=1.2)
    ax.plot(x, qd_data[start_idx:end_idx, i], 'r-', label=f'dqd{i+1}', linewidth=1.2)
    
    # Add labels and title
    ax.set_xlabel(x_label)
    ax.set_ylabel(r'rad $\cdot$ sec$^{-1}$')
    ax.set_title(f'Motor {i+1}')
    
    # Add legend
    ax.legend(loc='best', fontsize='small')
    
    # Add grid for readability
    ax.grid(True, alpha=0.3)
    
    # Calculate y-axis limits
    y_min = min(np.min(dq_data[start_idx:end_idx, i]), np.min(dqd_data[start_idx:end_idx, i]))
    y_max = max(np.max(dq_data[start_idx:end_idx, i]), np.max(dqd_data[start_idx:end_idx, i]))
    
    # Add padding
    y_range = y_max - y_min
    y_min -= 0.1 * y_range
    y_max += 0.1 * y_range
    
    ax.set_ylim(y_min, y_max)

plt.tight_layout()

# Save the combined plot
combined_path = os.path.join(plot_dir, "all_velocities.png")
plt.savefig(combined_path, dpi=300)
print(f"Saved plot to {combined_path}")
plt.close()


dq data shape: (10167, 10)
dqd data shape: (10167, 10)
Saved plot for dimension 1 to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/data/09_04_2025_data_NEA/04_16_2025_03_03_18_tetherless_A/plots/velocity_motor_1.png
Saved plot for dimension 2 to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/data/09_04_2025_data_NEA/04_16_2025_03_03_18_tetherless_A/plots/velocity_motor_2.png
Saved plot for dimension 3 to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/data/09_04_2025_data_NEA/04_16_2025_03_03_18_tetherless_A/plots/velocity_motor_3.png
Saved plot for dimension 4 to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/data/09_04_2025_data_NEA/04_16_2025_03_03_18_tetherless_A/plots/velocity_motor_4.png
Saved plot for dimension 5 to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/data/09_04_2025_data_NEA/04_16_2025_03_03_18_tetherless_A/plots/velocity_motor_5.png
Saved plot for dimension 6 to 

## Depth Tracking
### Depth and Desired Depth

In [28]:
depth_data = npz_file['depth']
# try:
#     depth_d_data = npz_file['depth_d']
#     print(depth_d_data[0:10])

# except:
print("no depth d, using default")
depth_d_data = np.ones(depth_data.shape) * 1.1
print(depth_d_data[0:10])
print(f"depth data shape: {depth_data.shape}")
print(f"depth d data shape: {depth_d_data.shape}")
# Define range to plot
start_idx = 0
end_idx = depth_data.shape[0]

# Prepare x-axis data
if 't' in npz_file:
    t_data = npz_file['t']
    if len(t_data) == len(depth_data):
        x = t_data[start_idx:end_idx]
        x_label = 'Time (seconds)'
    else:
        x = np.arange(start_idx, end_idx)
        x_label = 'Index'
else:
    x = np.arange(start_idx, end_idx)
    x_label = 'Index'

fig, ax = plt.subplots(figsize=(12, 7))

# Plot both q and dqd on same axis
ax.plot(x, depth_data[start_idx:end_idx], 'b-', label='Depth', linewidth=1.5)
ax.plot(x, depth_d_data[start_idx:end_idx], 'r-', label='Desired Depth', linewidth=1.5)

# Add labels and title
ax.set_xlabel(x_label)
ax.set_ylabel('Depth (meters)')
ax.set_title('Depth and Desired Depth')

# Add legend
ax.legend(loc='best')

# Add grid for readability
ax.grid(True, alpha=0.3)

# Calculate y-axis limits to ensure both datasets are visible
y_min = min(np.min(depth_data[start_idx:end_idx]), np.min(depth_d_data[start_idx:end_idx]))
y_max = max(np.max(depth_data[start_idx:end_idx]), np.max(depth_d_data[start_idx:end_idx]))

# Add some padding to the y-axis limits
y_range = y_max - y_min
y_min -= 0.1 * y_range
y_max += 0.1 * y_range

ax.set_ylim(y_min, y_max)

plt.tight_layout()

# Save the plot
save_path = os.path.join(plot_dir, f"depth_data.png")
plt.savefig(save_path, dpi=300)
print(f"Saved plot for depth to {save_path}")
plt.show()
# Close the plot to free memory
plt.close(fig)


# Calculate errors
depth_error = depth_data[start_idx:end_idx] - depth_d_data[start_idx:end_idx]
rmse = np.sqrt(np.mean(np.square(depth_error)))
mae = np.mean(np.abs(depth_error))
max_error = np.max(np.abs(depth_error))

# Add error metrics to plot
error_text = f"RMSE: {rmse:.4f} m\nMAE: {mae:.4f} m\nMax Error: {max_error:.4f} m"
print(error_text)

no depth d, using default
[1.1 1.1 1.1 1.1 1.1 1.1 1.1 1.1 1.1 1.1]
depth data shape: (21713,)
depth d data shape: (21713,)
Saved plot for depth to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/depth_data.png
RMSE: 0.4772 m
MAE: 0.3017 m
Max Error: 1.1400 m


In [20]:
import matplotlib.pyplot as plt
import numpy as np
from polish_matplotlib import polish_matplotlib  # Import the function

plt.style.use('default')
plt.rcParams['font.family'] = 'Nimbus Sans'

depth_data = npz_file['depth']
try:
    depth_d_data = npz_file['depth_d']
except:
    print("no depth d, using default")
depth_d_data = np.ones(depth_data.shape)

print(f"depth data shape: {depth_data.shape}")
print(f"depth d data shape: {depth_d_data.shape}")

# Define range to plot
start_idx = 0
end_idx = depth_data.shape[0]

# Prepare x-axis data
if 't' in npz_file:
    t_data = npz_file['t']
    if len(t_data) == len(depth_data):
        x = t_data[start_idx:end_idx]
        x_label = 'Time (s)'
    else:
        x = np.arange(start_idx, end_idx)
        x_label = 'Index'
else:
    x = np.arange(start_idx, end_idx)
    x_label = 'Index'

# Calculate error metrics FIRST (using raw data, same as old code)
depth_error = depth_data[start_idx:end_idx] - depth_d_data[start_idx:end_idx]
rmse = np.sqrt(np.mean(np.square(depth_error)))
mae = np.mean(np.abs(depth_error))
max_error = np.max(np.abs(depth_error))

# Colors matching the uploaded image
color_actual = '#4ECDC4'  # Teal/cyan for actual depth
color_desired = '#FF9966'  # Orange for desired depth

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), height_ratios=[2, 1])

# Top plot: Depth tracking
line1 = ax1.plot(x, depth_data[start_idx:end_idx], '-', color=color_actual, 
                 label='Actual Depth', linewidth=2, alpha=0.9, zorder=3)
line2 = ax1.plot(x, depth_d_data[start_idx:end_idx], '-', color=color_desired, 
                 label='Desired Depth', linewidth=2, alpha=0.9, zorder=2)

# Shade the error region with teal
ax1.fill_between(x, depth_data[start_idx:end_idx], depth_d_data[start_idx:end_idx],
                 alpha=0.2, color=color_actual, zorder=1)

ax1.set_ylabel('Depth (m)')
ax1.set_xlabel('')  # Remove xlabel from top plot
ax1.legend(loc='upper right', framealpha=0.9)

# Bottom plot: Tracking error
line3 = ax2.plot(x, depth_error, '-', color=color_actual, linewidth=2, alpha=0.9)
ax2.axhline(y=0, color='k', linestyle='-', linewidth=1, alpha=0.5)
ax2.fill_between(x, 0, depth_error, alpha=0.3, color=color_actual)

ax2.set_xlabel(x_label)
ax2.set_ylabel('Error (m)')

# Add error metrics as text box
error_text = f"RMSE: {rmse:.4f} m\nMAE: {mae:.4f} m\nMax: {max_error:.4f} m"
ax2.text(0.98, 0.95, error_text, transform=ax2.transAxes,
         fontsize=10, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, 
                   edgecolor='gray', linewidth=0.5))

# Apply polish_matplotlib styling
polish_matplotlib(fig, [ax1, ax2],
                 base_fontsize=12,      # Tick labels
                 label_fontsize=14,     # Axis labels
                 line_width=2,          # Line width
                 marker_size=8,
                 fig_size_inches=(10, 6),
                 use_tex=False,         # Set to True if you have LaTeX
                 grid=True,
                 box_on=True)

# Save the plot
save_path = os.path.join(plot_dir, f"depth_tracking.png")
plt.savefig(save_path, dpi=600, bbox_inches='tight', facecolor='white')
print(f"Saved plot to {save_path}")
print(error_text)
plt.close(fig)

depth data shape: (21713,)
depth d data shape: (21713,)
Saved plot to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/depth_tracking.png
RMSE: 0.4551 m
MAE: 0.3264 m
Max: 1.0400 m


## Stereo Depth Images
### stereo depth calculated during experiment

In [24]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Set up seaborn-like styling manually
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'  # Light purple-blue background
plt.rcParams['axes.edgecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#EAEAF2'
plt.rcParams['grid.linewidth'] = 1
plt.rcParams['axes.axisbelow'] = True
plt.rcParams['font.size'] = 11

# Seaborn default color palette
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', 
          '#937860', '#DA8BC3', '#8C8C8C', '#CCB974', '#64B5CD']

stereo_depth_data = npz_file['stereo_debug_depth']
turn_commands = npz_file['turn_commands']
print(stereo_depth_data.shape)

t_data = t_quat_heading[:stereo_depth_data.shape[0]]
print(stereo_depth_data[6650:7300])

start_idx = 0
end_idx = stereo_depth_data.shape[0]
print(stereo_depth_data.shape)

# Create figure
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare x-axis data
if len(t_data) == len(stereo_depth_data):
    x = t_data[start_idx:end_idx]
    x_label = 'Time'
    print("doing this")
    print(f"x: {x}")
else:
    x = np.arange(start_idx, end_idx)
    x_label = 'Index'

indices = x

# Plot with seaborn colors
ax.plot(x, stereo_depth_data[start_idx:end_idx], linewidth=2.5, 
        color=colors[0], alpha=0.9, label='Depth')

# Add turn command shading
in_turn = False
turn_start = None
turn_type = None
left_labeled = False
right_labeled = False

for i in range(start_idx, end_idx):
    if i < len(turn_commands):
        cmd = turn_commands[i]
        
        # Starting a turn
        if not in_turn and cmd in ["turn left", "turn right"]:
            in_turn = True
            turn_start = indices[i - start_idx]
            turn_type = cmd
        
        # Ending a turn
        elif in_turn and cmd not in ["turn left", "turn right"]:
            turn_end = indices[i - start_idx]
            
            # Add colored background with seaborn colors
            if turn_type == "turn left":
                ax.axvspan(turn_start, turn_end, alpha=0.25, color=colors[2],  # Green
                          label='Turn Left' if not left_labeled else '', zorder=1)
                left_labeled = True
            elif turn_type == "turn right":
                ax.axvspan(turn_start, turn_end, alpha=0.25, color=colors[1],  # Orange
                          label='Turn Right' if not right_labeled else '', zorder=1)
                right_labeled = True
            
            in_turn = False
            turn_type = None

# Handle case where turn extends to end of plot
if in_turn:
    if turn_type == "turn left":
        ax.axvspan(turn_start, indices[-1], alpha=0.25, color=colors[2],
                  label='Turn Left' if not left_labeled else '', zorder=1)
    elif turn_type == "turn right":
        ax.axvspan(turn_start, indices[-1], alpha=0.25, color=colors[1],
                  label='Turn Right' if not right_labeled else '', zorder=1)

# Customize plot with seaborn-style aesthetics
ax.set_xlabel(x_label, fontsize=12, fontweight='normal')
ax.set_ylabel('Depth (meters)', fontsize=12, fontweight='normal')
ax.set_ylim(0, 10)
ax.set_title('Stereo Depth', fontsize=14, fontweight='normal', pad=20)

# Legend styling
legend = ax.legend(loc='best', frameon=True, framealpha=0.9, 
                   edgecolor='white', fancybox=False)
legend.get_frame().set_facecolor('white')

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('white')
ax.spines['bottom'].set_color('white')

# Style tick labels
ax.tick_params(colors='#555555', which='both')

plt.tight_layout()

# Save the plot
save_path = os.path.join(plot_dir, f"stereo_depth_debug_plot.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved plot for stereo depth to {save_path}")
plt.close(fig)

(8422,)
[3.54135251 3.54135251 4.43414307 4.43414307 4.43414307 4.43414307
 4.43414307 4.17231464 4.17231464 4.17231464 4.17231464 4.17231464
 4.17231464 4.17231464 4.17231464 4.17231464 4.17231464 4.26227379
 4.26227379 4.26227379 4.26227379 4.26227379 4.26227379 4.59239244
 4.59239244 4.59239244 4.59239244 4.59239244 4.59239244 4.59239244
 5.20831394 5.20831394 5.20831394 5.20831394 5.20831394 5.20831394
 5.20831394 5.20831394 5.20831394 5.20831394 5.20831394 5.20831394
 5.20831394 5.20831394 5.20831394 5.20831394 5.20831394 5.20831394
 5.20831394 5.20831394 5.20831394 5.20831394 4.9552474  4.9552474
 4.9552474  4.9552474  4.9552474  4.9552474  4.9552474  4.41638803
 4.41638803 4.41638803 4.41638803 4.41638803 4.41638803 4.41638803
 4.41638803 4.41638803 4.41638803 4.41638803 4.41638803 4.41638803
 4.41638803 4.33593798 4.33593798 4.33593798 4.33593798 4.33593798
 5.75834131 5.75834131 5.75834131 5.75834131 5.75834131 5.75834131
 5.79769039 5.79769039 5.79769039 5.79769039 5.79769039

In [21]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('default')
plt.rcParams['font.family'] = 'Nimbus Sans'
stereo_depth_data = npz_file['stereo_debug_depth']
turn_commands = npz_file['turn_commands']
t_data = t_quat_heading[:stereo_depth_data.shape[0]]

start_idx = 0
end_idx = stereo_depth_data.shape[0]

# Use larger figure with better aspect ratio for papers
fig, ax = plt.subplots(figsize=(10, 4))

# x = t_data[start_idx:end_idx]
x = np.arange(start_idx, end_idx)

depth = stereo_depth_data[start_idx:end_idx]

# Filter out zero/invalid depths for cleaner visualization
valid_mask = depth > 0.1
x_valid = x[valid_mask]
depth_valid = depth[valid_mask]

# Highlight turn regions FIRST (so they're behind the data)
in_turn = False
turn_start = None
turn_type = None
left_labeled = False
right_labeled = False

for i in range(start_idx, end_idx):
    if i < len(turn_commands):
        cmd = turn_commands[i]
        
        if not in_turn and cmd in ["turn left", "turn right"]:
            in_turn = True
            turn_start = x[i - start_idx]
            turn_type = cmd
        
        elif in_turn and cmd not in ["turn left", "turn right"]:
            turn_end = x[i - start_idx]
            
            if turn_type == "turn left":
                ax.axvspan(turn_start, turn_end, alpha=0.15, color='#4ECDC4',
                          label='Turn Left' if not left_labeled else '', zorder=1)
                left_labeled = True
            elif turn_type == "turn right":
                ax.axvspan(turn_start, turn_end, alpha=0.15, color='#FFD93D',
                          label='Turn Right' if not right_labeled else '', zorder=1)
                right_labeled = True
            
            in_turn = False
            turn_type = None

if in_turn:
    if turn_type == "turn left":
        ax.axvspan(turn_start, x[-1], alpha=0.15, color='#4ECDC4',
                  label='Turn Left' if not left_labeled else '', zorder=1)
    elif turn_type == "turn right":
        ax.axvspan(turn_start, x[-1], alpha=0.15, color='#FFD93D',
                  label='Turn Right' if not right_labeled else '', zorder=1)

# Plot the depth data on top
ax.plot(x_valid, depth_valid, linewidth=1.2, color='#2C3E50', zorder=3, alpha=0.8)

# Styling for publication
ax.set_xlabel('Time (s)', fontsize=11, fontweight='normal')
ax.set_ylabel('Obstacle Distance (m)', fontsize=11, fontweight='normal')
# ax.set_title('Stereo Depth Measurements During Autonomous Navigation', 
#              fontsize=12, fontweight='bold', pad=15)

# Better axis limits
ax.set_ylim(0, 8)  # Reduced from 10 since data doesn't go that high
ax.set_xlim(x[0], x[-1])

# Cleaner grid
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5, zorder=0)
ax.set_axisbelow(True)

# Better legend
ax.legend(loc='center', framealpha=0.9, fontsize=10, 
         edgecolor='gray', fancybox=False)

# Remove top and right spines for cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Tighter layout
plt.tight_layout()

# Save with higher DPI for publication
save_path = os.path.join(plot_dir, f"stereo_depth_publication.png")
plt.savefig(save_path, dpi=600, bbox_inches='tight', facecolor='white')
print(f"Saved publication-quality plot to {save_path}")
plt.close(fig)

Saved publication-quality plot to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/stereo_depth_publication.png


## Depth Turn Plot -- FOR PAPER

In [23]:
import matplotlib.pyplot as plt
import numpy as np

stereo_depth_data = npz_file['stereo_debug_depth']
turn_commands = npz_file['turn_commands']
t_data = t_quat_heading[:stereo_depth_data.shape[0]]

start_idx = 0
end_idx = stereo_depth_data.shape[0]

# Use larger figure with better aspect ratio for papers
fig, ax = plt.subplots(figsize=(10, 4))

x = t_data[start_idx:end_idx]
depth = stereo_depth_data[start_idx:end_idx]

# Filter out zero/invalid depths for cleaner visualization
valid_mask = depth > 0.1
x_valid = x[valid_mask]
depth_valid = depth[valid_mask]

# Highlight turn regions FIRST (so they're behind the data)
in_turn = False
turn_start = None
turn_type = None
left_labeled = False
right_labeled = False
threshold = 3.0

for i in range(start_idx, end_idx):
    if i < len(turn_commands):
        cmd = turn_commands[i]
        
        if not in_turn and cmd in ["turn left", "turn right"]:
            in_turn = True
            turn_start = x[i - start_idx]
            turn_type = cmd
        
        elif in_turn and cmd not in ["turn left", "turn right"]:
            turn_end = x[i - start_idx]
            
            if turn_type == "turn left":
                ax.fill_between([turn_start, turn_end], 0, threshold, alpha=0.15, color='#4ECDC4',
                               label='Turn Left' if not left_labeled else '', zorder=1)
                # ax.axvspan(turn_start, turn_end, alpha=0.15, color='#4ECDC4',
                #           label='Turn Left' if not left_labeled else '', zorder=1)
                left_labeled = True
            elif turn_type == "turn right":
                # ax.axvspan(turn_start, turn_end, alpha=0.15, color='#FFD93D',
                #           label='Turn Right' if not right_labeled else '', zorder=1)
                ax.fill_between([turn_start, turn_end], 0, threshold, alpha=0.15, color='#FFD93D',
                               label='Turn Right' if not right_labeled else '', zorder=1)

                right_labeled = True
            
            in_turn = False
            turn_type = None

if in_turn:
    if turn_type == "turn left":
        # ax.axvspan(turn_start, x[-1], alpha=0.15, color='#4ECDC4',
        #           label='Turn Left' if not left_labeled else '', zorder=1)
        ax.fill_between([turn_start, x[-1]], 0, threshold, alpha=0.15, color='#4ECDC4',
                       label='Turn Left' if not left_labeled else '', zorder=1)

    elif turn_type == "turn right":
        # ax.axvspan(turn_start, x[-1], alpha=0.15, color='#FFD93D',
        #           label='Turn Right' if not right_labeled else '', zorder=1)
        ax.fill_between([turn_start, x[-1]], 0, threshold, alpha=0.15, color='#FFD93D',
                       label='Turn Right' if not right_labeled else '', zorder=1)

# Add threshold line
ax.axhline(y=threshold, color='#E74C3C', linestyle='--', linewidth=2, 
           label='Avoidance Threshold (3.0 m)', zorder=2, alpha=0.8)

# Plot the depth data on top
# ax.plot(x_valid, depth_valid, linewidth=1.2, color='#2C3E50', 
#         zorder=3, alpha=0.8, label='Measured Distance')
ax.plot(x_valid, depth_valid, linewidth=1.2, color='black',
        zorder=3, alpha=0.8, label='Measured Distance')

# Styling for publication
ax.set_xlabel('Time (s)', fontsize=11, fontweight='normal')
ax.set_ylabel('Obstacle Distance (m)', fontsize=11, fontweight='normal')
# ax.set_title('Stereo Depth Measurements During Autonomous Navigation', 
#              fontsize=12, fontweight='bold', pad=15)

# Better axis limits
ax.set_ylim(0, 10)
ax.set_xlim(x[0], x[-1])

# Cleaner grid
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5, zorder=0)
ax.set_axisbelow(True)

# Better legend
# ax.legend(loc='upper', framealpha=0.9, fontsize=10, 
#          edgecolor='gray', fancybox=False)
ax.legend(loc='upper center', bbox_to_anchor=(0.52, 0.9), 
          framealpha=0.9, fontsize=10, edgecolor='gray', fancybox=False)
# Remove top and right spines for cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

save_path = os.path.join(plot_dir, f"stereo_depth_publication.jpeg")
plt.savefig(save_path, dpi=600, bbox_inches='tight', facecolor='white')
print(f"Saved publication-quality plot to {save_path}")
plt.close(fig)

Saved publication-quality plot to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/stereo_depth_publication.jpeg


## Nav U Plotting

#### [velocity, roll, pitch for front flippers, yaw, pitch for rear flippers]


In [ ]:
nav_u_data = npz_file['nav_u']
nav_u_data = np.clip(nav_u_data, -1.0, 1.0)

start_idx = 0
end_idx = nav_u_data.shape[0]

# Create 4 subplots stacked vertically
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

x = np.arange(start_idx, end_idx)
labels = ['Velocity', 'Roll', 'Pitch', 'Yaw']

for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.plot(x, nav_u_data[start_idx:end_idx, i])
    ax.set_ylabel(f'{label} Command')
    ax.set_ylim(-1.1, 1.1)
    ax.grid(True)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)

# Only set xlabel on bottom plot
axes[-1].set_xlabel('Index')

# Add overall title
fig.suptitle('Nav U Commands', fontsize=14)
plt.tight_layout()

save_path = os.path.join(plot_dir, f"nav_u_all_axes.png")
plt.savefig(save_path, dpi=300)
print(f"Saved plot to {save_path}")
plt.close(fig)

Saved plot to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/nav_u_all_axes.png


## Yaw Tracking

### Measured and Desired Yaw

In [ ]:
yaw_d = npz_file['yaw_d'][1:]
headings = npz_file['headings']
print(f"yaw_d shape: {yaw_d.shape}")
print(f"headings shape: {headings.shape}")
# index 481 is when we get our first turn command   
start_idx = 0
end_idx = 500

# Create a nice-looking plot
plt.figure(figsize=(14, 8))

# # Check if time data is available
# if 't' in npz_file and len(npz_file['t']) >= len(quat):
#     # Use time as x-axis if available
#     t = npz_file['t'][:len(quat)]
#     plt.plot(t, headings, 'b-', linewidth=2, label='Actual Yaw')
#     plt.plot(t, yaw_d, 'r-', linewidth=2, label='Desired Yaw')
#     plt.xlabel('Time')
# else:
# Otherwise use indices
# indices = np.arange(len(headings))
indices = np.arange(start_idx, end_idx)
indices = t_quat_heading[start_idx: end_idx]
plt.plot(indices, headings[start_idx:end_idx], 'b-', linewidth=2, label='Actual Yaw')
plt.plot(indices, yaw_d[start_idx:end_idx], 'r-', linewidth=2, label='Desired Yaw')
plt.xlabel('Sample Index')

# Set labels and title
plt.ylabel('Yaw Angle')
plt.title('Comparison of Actual vs Desired Yaw Angle')

# Add grid and legend
plt.grid(True, alpha=0.3)
plt.legend(loc='best')

# Calculate y-axis limits to ensure both datasets are visible
y_min = min(np.min(headings), np.min(headings))
y_max = max(np.max(headings), np.max(headings))

# Add some padding to the y-axis limits
y_range = y_max - y_min
y_min -= 0.1 * y_range
y_max += 0.1 * y_range

plt.ylim(y_min, y_max)

# Tight layout for better appearance
plt.tight_layout()
# Save the plot
save_path = os.path.join(plot_dir, f"yaw_vs_yaw_desired.png")
# Save the plot if a path is provided
if save_path:
    plt.savefig(save_path, dpi=300)
    print(f"Plot saved to: {save_path}")

# # Create an additional plot showing the error between actual and desired yaw
# plt.figure(figsize=(14, 6))

# # Calculate yaw error
# yaw_error = headings - yaw_d

# # Plot the error
# if 't' in npz_file and len(npz_file['t']) >= len(quat):
#     t = npz_file['t'][:len(yaw_error)]
#     plt.plot(t, yaw_error, 'g-', linewidth=2)
#     plt.xlabel('Time')
# else:
#     plt.plot(indices, yaw_error, 'g-', linewidth=2)
#     plt.xlabel('Sample Index')

# plt.ylabel('Yaw Error (Actual - Desired)')
# plt.title('Yaw Tracking Error')
# plt.grid(True, alpha=0.3)

# # Add a horizontal line at zero for reference
# plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)

# plt.tight_layout()

# # Save the error plot if a path is provided
# if save_path:
#     error_path = save_path.replace('.png', '_error.png')
#     plt.savefig(error_path, dpi=300)
#     print(f"Error plot saved to: {error_path}")

# # Calculate and print some statistics about the tracking performance
# mean_error = np.mean(yaw_error)
# rmse = np.sqrt(np.mean(yaw_error**2))
# max_error = np.max(np.abs(yaw_error))

# print("\nYaw Tracking Performance:")
# print(f"Mean Error: {mean_error:.4f}")
# print(f"RMSE: {rmse:.4f}")
# print(f"Maximum Absolute Error: {max_error:.4f}")

# Show the plots
# plt.show()



yaw_d shape: (8422,)
headings shape: (8422,)
[2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2
 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2
 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2
 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2
 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2
 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2 2.2]
Plot saved to: /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/yaw_vs_yaw_desired.png


In [55]:
yaw_d = npz_file['yaw_d'][1:]
headings = npz_file['headings']
turn_commands = npz_file['turn_commands']  # Load turn commands

print(f"yaw_d shape: {yaw_d.shape}")
print(f"headings shape: {headings.shape}")
print(f"turn_commands shape: {len(turn_commands)}")

start_idx = 6000
end_idx = 8000

# Create a nice-looking plot
plt.figure(figsize=(14, 8))

indices = t_quat_heading[start_idx:end_idx]

# Plot yaw data
plt.plot(indices, headings[start_idx:end_idx], 'b-', linewidth=2, label='Actual Yaw', zorder=3)
# plt.plot(indices, yaw_d[start_idx:end_idx], 'r-', linewidth=2, label='Desired Yaw', zorder=3)

# Highlight turn regions
# Assuming turn_commands has same length as headings
# Highlight turn regions
in_turn = False
turn_start = None
turn_type = None
left_labeled = False
right_labeled = False

for i in range(start_idx, end_idx):
    if i < len(turn_commands):
        cmd = turn_commands[i]
        
        # Starting a turn
        if not in_turn and cmd in ["turn left", "turn right"]:
            in_turn = True
            turn_start = indices[i - start_idx]
            turn_type = cmd
        
        # Ending a turn
        elif in_turn and cmd not in ["turn left", "turn right"]:
            turn_end = indices[i - start_idx]
            
            # Add colored background
            if turn_type == "turn left":
                plt.axvspan(turn_start, turn_end, alpha=0.2, color='cyan', 
                           label='Turn Left' if not left_labeled else '', zorder=1)
                left_labeled = True
            elif turn_type == "turn right":
                plt.axvspan(turn_start, turn_end, alpha=0.2, color='yellow', 
                           label='Turn Right' if not right_labeled else '', zorder=1)
                right_labeled = True
            
            in_turn = False
            turn_type = None

# Handle case where turn extends to end of plot
if in_turn:
    if turn_type == "turn left":
        plt.axvspan(turn_start, indices[-1], alpha=0.2, color='cyan', 
                   label='Turn Left' if not left_labeled else '', zorder=1)
    elif turn_type == "turn right":
        plt.axvspan(turn_start, indices[-1], alpha=0.2, color='yellow', 
                   label='Turn Right' if not right_labeled else '', zorder=1)
plt.xlabel('Time (s)')
plt.ylabel('Yaw Angle')
plt.title('Comparison of Actual vs Desired Yaw Angle')
plt.grid(True, alpha=0.3, zorder=1)
plt.legend(loc='best')

# Calculate y-axis limits
y_min = min(np.min(headings[start_idx:end_idx]), np.min(yaw_d[start_idx:end_idx]))
y_max = max(np.max(headings[start_idx:end_idx]), np.max(yaw_d[start_idx:end_idx]))
y_range = y_max - y_min
y_min -= 0.1 * y_range
y_max += 0.1 * y_range
plt.ylim(y_min, y_max)

plt.tight_layout()

save_path = os.path.join(plot_dir, f"yaw_vs_yaw_desired.png")
plt.savefig(save_path, dpi=300)
print(f"Plot saved to: {save_path}")
plt.close()

yaw_d shape: (8422,)
headings shape: (8422,)
turn_commands shape: 8422
Plot saved to: /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/09_04_2025_logs_NEA/20250416_031346_tetherless_C/plots/yaw_vs_yaw_desired.png


## Centroid Plots

In [ ]:
import numpy as np
from matplotlib import cm

def recover_class_labels(overlay, img_to_viz, num_of_labels, alpha=0.5, cmap=None, return_viz=False):
    """
    Invert multiclass_vis:
      overlay = alpha * cmap(class_labels/num_of_labels)[..., :3] + (1 - alpha) * (img_to_viz/255)

    Args:
        overlay: HxWx3 float in [0,1] or uint8 in [0,255] (the function's returned _overlay).
        img_to_viz: HxWx3 uint8 original image used in multiclass_vis.
        num_of_labels: number of classes (K).
        alpha: blend factor used originally.
        cmap: matplotlib colormap; if None, uses 'viridis' (must match what was used originally).
        return_viz: if True, also return the recovered colormap image (viz_est).

    Returns:
        class_labels: HxW int array with values in [0, K-1].
        (optional) viz_est: HxWx3 float in [0,1], the recovered colormap RGB per pixel.
    """
    if cmap is None:
        cmap = cm.get_cmap('viridis')

    # Ensure float32 in [0,1]
    over = overlay.astype(np.float32)
    if over.max() > 1.0:
        over /= 255.0
    base = img_to_viz.astype(np.float32) / 255.0

    eps = 1e-8
    if alpha <= eps:
        raise ValueError("alpha must be > 0 to invert the blend.")

    # Recover the colormap image that was blended in
    viz_est = (over - (1.0 - alpha) * base) / alpha
    viz_est = np.clip(viz_est, 0.0, 1.0)

    # Build the discrete palette used by cmap(class_labels/num_of_labels)
    t = (np.arange(num_of_labels, dtype=np.float32) / float(num_of_labels))
    palette = cmap(t)[..., :3].astype(np.float32)  # Kx3, RGB

    # Assign each pixel to nearest palette color (Euclidean in RGB)
    H, W, _ = viz_est.shape
    v = viz_est.reshape(-1, 3)                        # (N,3)
    # (N,K,3) -> (N,K)
    d2 = ((v[:, None, :] - palette[None, :, :]) ** 2).sum(axis=2)
    labels = d2.argmin(axis=1).astype(np.int32).reshape(H, W)

    if return_viz:
        return labels, viz_est
    return labels


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming image1.jpg is the overlay and image2.jpg is the original image

overlay_img = plt.imread('/image2.jpg')
original_img = plt.imread('/image1.jpg')



# You need to provide the correct number of labels and alpha used to create the overlay
# Replace these with the actual values
num_of_labels = 2
alpha = 0.5

class_labels = recover_class_labels(overlay_img, original_img, num_of_labels, alpha=alpha)

print("Recovered class labels shape:", class_labels.shape)
# Display the recovered class labels (optional)
plt.imshow(class_labels, cmap='viridis')
plt.colorbar()
plt.title("Recovered Class Labels")
plt.show()